### Encoder and Decoder in Transformers

#### Encoder
- The encoder processes the input sequence and generates a sequence of continuous representations (embeddings).
- It captures contextual information from the input by attending to all positions in the sequence using self-attention mechanisms.
- The encoder is typically used in tasks where the entire input sequence is required to understand the context, such as text classification or machine translation.

#### Decoder
- The decoder generates an output sequence based on the encoded representations from the encoder and previously generated tokens.
- It uses self-attention to focus on the output sequence generated so far and cross-attention to attend to the encoder's output.
- The decoder is autoregressive, meaning it generates one token at a time, conditioning on previously generated tokens.

---

### Tasks That Use Both Encoder and Decoder
Tasks that require both an input sequence and an output sequence typically use the full encoder-decoder architecture. Examples include:
- **Machine Translation**: Translating text from one language to another.
- **Summarization**: Generating a concise summary of a given text.
- **Question Answering (Generative)**: Producing a natural language answer based on a given context and question.

---

### Tasks That Use Decoder-Only
Decoder-only architectures are used in tasks where the model generates output without requiring a separate input sequence or where the input is embedded into the same sequence as the output. Examples include:
- **Text Generation**: Generating coherent text, such as stories or articles.
- **Autoregressive Language Modeling**: Predicting the next token in a sequence (e.g., GPT models).
- **Code Generation**: Writing code based on a prompt or partial code snippet.
- **Chatbots**: Generating conversational responses in dialogue systems.

In decoder-only models, the input and output are often combined into a single sequence, and the model learns to predict the next token in the sequence.

![encoder-decoder](./resources/encoder-decoder.png)

We will make use of the encoder that we built previously, and write a decoder part of it.

In [ ]:
import torch
from torch import nn

class EncoderBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        d_ff: int = None,
        dropout: float = 0.1
    ):
        super().__init__()
        # The same as the attention that we talked about before
        # but pytorch has it ready and we don't need to implement
        # on our own
        self.attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm((d_model, ))
        # A convention is to have 4 * d_model as the output shape
        d_ff = d_model * 4 if not d_ff else d_ff
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm2 = nn.LayerNorm((d_model, ))
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, X, padding_mask):
        """X has shape (batch_size, sequence_length, d_model)"""
        # Apply multi-head attention to get new representation
        attn_output, _ = self.attention(X, X, X, key_padding_mask=padding_mask)
        # Add & Norm for the multi-head attention output
        norm1_input = X + attn_output
        norm1_output = self.norm1(norm1_input)
        # Feed forward
        ffn_output = self.linear_relu_stack(norm1_output)
        ffn_output = self.dropout2(ffn_output)
        # Add & Norm for the FFN output
        norm2_input = norm1_output + ffn_output
        norm2_output = self.norm2(norm2_input)
        return norm2_output

In [ ]:
import torch
from torch import nn

class DecoderBlock(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        d_ff: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        # The masked Multi-head attention
        self.masked_self_attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm1 = nn.LayerNorm((d_model, ))
        self.dropout1 = nn.Dropout(dropout)
        # The attention that came from the encoder
        self.encoder_decoder_attention = nn.MultiheadAttention(
            embed_dim=d_model,
            num_heads=heads,
            dropout=dropout,
            batch_first=True,
        )
        self.norm2 = nn.LayerNorm((d_model, ))
        self.dropout2 = nn.Dropout(dropout)
        # The feed forward (FFN) part
        # A convention is to have 4 * d_model as the output shape
        d_ff = d_model * 4 if not d_ff else d_ff
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )
        self.norm3 = nn.LayerNorm(d_model) # Normalizes over the d_model dimension
        self.dropout3 = nn.Dropout(dropout) # Dropout after FFN residual

    def forward(self, target_input, encoder_output, target_mask=None, encoder_mask=None):
        """
        Decoder will need to take in the encoder output as part of the input
        """
        residual_input = target_input
        # --- Masked Multi-Head Self-Attention ---
        # Query, Key, Value are the same (from the decoder's path)
        # Pass the target_mask (causal + padding if needed)
        masked_attn_output, _ = self.masked_self_attention(
            query=target_input,
            key=target_input,
            value=target_input,
            attn_mask=target_mask # Apply the mask here
        )

        # --- Add & Norm 1 ---
        # Add residual connection (input to this sub-layer)
        output_after_self_attn = residual_input + masked_attn_output
        # Apply dropout
        output_after_self_attn = self.dropout1(output_after_self_attn)
        # Apply Layer Norm
        norm1_output = self.norm1(output_after_self_attn) # Shape (batch_size, target_seq_len, d_model)

        # Store input for next residual connection
        residual_input = norm1_output # Input to encoder-decoder attention

        # --- Multi-Head Encoder-Decoder Attention (Cross-Attention) ---
        # Query from decoder's path (output of first norm)
        # Key and Value from encoder's output
        # encoder_mask can be used here to mask padding in the encoder output
        cross_attn_output, _ = self.encoder_decoder_attention(
            query=norm1_output,
            key=encoder_output,
            value=encoder_output,
            attn_mask=encoder_mask # Apply encoder padding mask here if needed
        )

        # --- Add & Norm 2 ---
        # Add residual connection (input to this sub-layer)
        output_after_cross_attn = residual_input + cross_attn_output
        # Apply dropout
        output_after_cross_attn = self.dropout2(output_after_cross_attn)
        # Apply Layer Norm
        norm2_output = self.norm2(output_after_cross_attn) # Shape (batch_size, target_seq_len, d_model)

        # Store input for next residual connection
        residual_input = norm2_output # Input to Feed-Forward Network

        # --- Feed-Forward Network ---
        # FFN operates independently on the last dimension
        ffn_output = self.linear_relu_stack(norm2_output) # Shape (batch_size, target_seq_len, d_model)
        # Apply dropout
        ffn_output = self.dropout3(ffn_output) # Dropout after FFN output

        # --- Add & Norm 3 ---
        # Add residual connection (input to this sub-layer)
        output_after_ffn = residual_input + ffn_output
        # Apply Layer Norm
        norm3_output = self.norm3(output_after_ffn) # Shape (batch_size, target_seq_len, d_model)

        return norm3_output # Output of the Decoder Block

The final step is to combine the encoder with the decoder

In [ ]:
from torch import nn
import torch
import math

class EncoderDecoderTransformer(nn.Module):
    def __init__(
        self,
        d_model: int,
        heads: int,
        vocab_size_source: int,
        vocab_size_target: int,
        num_encoder_layers: int,
        num_decoder_layers: int,
        max_seq_len: int,
        d_ff: int = None,
        dropout: float = 0.1,
    ):
        super().__init__()
        self.dropout = dropout
        self.d_model = d_model
        # Embeddings
        self.source_embeddings = nn.Embedding(vocab_size_source, d_model)
        self.target_embeddings = nn.Embedding(vocab_size_target, d_model)
        self.positional_encoding = self._generate_fixed_positional_encoding(max_seq_len, d_model)
        # Encoder layers
        self.encoder_stack = nn.ModuleList([
            EncoderBlock(d_model, heads, d_ff, dropout) for _ in range(num_encoder_layers)
        ])
        # Decoder layers
        self.decoder_stack = nn.ModuleList([
            DecoderBlock(d_model, heads, d_ff, dropout) for _ in range(num_decoder_layers)
        ])
        # Output layer
        self.output_layer = nn.Linear(d_model, vocab_size_target)
        # Initialize weights
        self._initialize_parameters()

    def _initialize_parameters(self):
        # Common initialization for Transformer weights
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def _generate_fixed_positional_encoding(self, max_seq_len: int, d_model: int):
        """Generate a matrix for fixed positional encoding base on the max_seq_len"""
        pe = torch.zeros(max_seq_len, d_model)
        position = torch.arange(0, max_seq_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # Add batch dimension (1, max_seq_len, d_model)
        return pe # This will be added to embeddings

    def forward(self, src_tokens, trg_tokens, encoder_padding_mask=None, decoder_padding_mask=None):
        """
        This is a high-level forward pass outline. Actual implementation needs mask handling.
        Masks need to be generated based on src_tokens and trg_tokens padding
        Lookahead mask for decoder self-attention also needs to be generated.
        """
        # 1. Get embeddings and add positional encoding
        src_embed = self.source_embeddings(src_tokens) * math.sqrt(self.d_model) # Scale embeddings
        trg_embed = self.target_embeddings(trg_tokens) * math.sqrt(self.d_model) # Scale embeddings

        # Add positional encoding (need to handle sequence length correctly)
        src_embed = src_embed + self.positional_encoding[:, :src_tokens.size(1), :].to(src_embed.device)
        trg_embed = trg_embed + self.positional_encoding[:, :trg_tokens.size(1), :].to(trg_embed.device)

        # Apply dropout (common)
        src_embed = nn.Dropout(self.dropout)(src_embed)
        trg_embed = nn.Dropout(self.dropout)(trg_embed)

        # 2. Encoder Pass
        encoder_output = src_embed
        for encoder_layer in self.encoder_stack:
            # Pass padding mask to encoder self-attention
            encoder_output = encoder_layer(encoder_output, src_mask=encoder_padding_mask)

        # 3. Decoder Pass
        decoder_output = trg_embed
        # Generate causal mask for decoder self-attention (shape seq_len, seq_len)
        causal_mask = torch.nn.Transformer.generate_square_subsequent_mask(trg_tokens.size(1)).to(trg_tokens.device)
        # Combine causal mask with target padding mask if needed

        for decoder_layer in self.decoder_stack:
            # Pass target causal mask to masked self-attention
            # Pass encoder padding mask to encoder-decoder attention
            decoder_output = decoder_layer(
                decoder_output,
                encoder_output,
                target_mask=causal_mask, # Pass the causal mask
                encoder_mask=encoder_padding_mask # Pass the encoder padding mask
            )

        # 4. Final Output Layer
        # Project d_model to vocab_size
        output_logits = self.output_layer(decoder_output)

        return output_logits # Shape (batch_size, target_sequence_length, vocab_size_target)